# USD/IDR: EDA dan deteksi regime Hamilton

## tl;dr
Snapshot `IDR=X` mencakup 1 Juli 2016–24 Juli 2026. Model Markov Switching dua-state mengidentifikasi kondisi high-volatility secara endogen. Probabilitas *smoothed* di notebook ini hanya bersifat retrospektif; fitur prediktif pada notebook berikutnya memakai probabilitas *filtered* yang dilag satu hari.

## Context & Methods

**Pertanyaan.** Apakah return USD/IDR membentuk state volatilitas berbeda tanpa memberi label manual pada tanggal krisis?

**Model.** `MarkovRegression` dua state pada log return harian (%), intercept state-specific, `switching_variance=True`. State high-vol dipilih dari realised variance state yang lebih tinggi—bukan dari nomor state.

**Asumsi penting.** Event Maret 2018, Maret 2020, dan Maret 2022 hanya annotation ekonomi. Mereka tidak dipakai sebagai label dan tidak membuktikan sebab-akibat.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import build_daily_features, chronological_split, load_market_snapshot
from src.hamilton_regime import fit_hamilton_smoothed, fit_hamilton_train_and_filter, regime_summary
from src.forecasting import plot_regime_probability

RAW_PATH = PROJECT_ROOT / 'data/raw/yahoo_usd_idr_us10y.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

## Data

Snapshot disimpan di repository agar notebook tidak tergantung unduhan live. Manifest mencatat checksum dan waktu pengambilan.

In [2]:
market = load_market_snapshot(RAW_PATH)
features = build_daily_features(market)
train, test = chronological_split(features)

quality_check = {
    'raw_rows': len(market),
    'raw_date_start': market.date.min().date(),
    'raw_date_end': market.date.max().date(),
    'duplicate_dates': int(market.date.duplicated().sum()),
    'non_positive_usd_idr': int((market.usd_idr <= 0).sum()),
    'feature_rows': len(features),
    'train_end': train.date.max().date(),
    'test_start': test.date.min().date(),
}
quality_check

## Results

Fit pertama hanya memakai train. Parameter fit itu kemudian dipakai untuk memfilter seluruh sampel. Fit kedua memakai seluruh sampel dan hanya menghasilkan visualisasi *smoothed*.

In [3]:
all_returns = features.set_index('date').log_return_pct
train_returns = train.set_index('date').log_return_pct
_, filtered_probability, filtered_high_state = fit_hamilton_train_and_filter(train_returns, all_returns)
_, smoothed_probability, smoothed_high_state = fit_hamilton_smoothed(all_returns)

regime_frame = features[['date', 'log_return_pct']].copy()
regime_frame['high_vol_probability_filtered'] = filtered_probability.to_numpy()
regime_frame['high_vol_probability_smoothed'] = smoothed_probability.to_numpy()
summary = regime_summary(all_returns, smoothed_probability)
summary

In [4]:
plot_regime_probability(regime_frame, OUTPUT_DIR / 'regime_probability.png')
from IPython.display import Image, display
display(Image(filename=str(OUTPUT_DIR / 'regime_probability.png')))

<IPython.core.display.Image object>


## Takeaways

- State dengan probabilitas high-vol besar memang memiliki absolute return dan realised variance yang lebih tinggi pada snapshot ini.
- Puncak probabilitas di sekitar beberapa event bisa digunakan sebagai pemeriksaan *economic plausibility*, bukan sebagai validasi kausal.
- Untuk forecasting, jangan gunakan seri *smoothed*: seri itu dibangun dengan informasi sesudah tanggal prediksi.